In [1]:
import numpy as np, os, h5py
import cupy as cp
import matplotlib.pyplot as plt

import astropy.units as u
import astropy.constants as cst
from astropy.cosmology import Planck18 as cosmo

from tqdm import tqdm
from moviepy.editor import ImageSequenceClip

from utils_cosmo import displacement_field, power_spectrum

## Initial Conditions for a Cosmological N-body Simulation

Before running the simulation, we must specify initial conditions consistent with the early universe:

### 1. Uniform Particle Grid
We distribute particles on a regular 3 D cubic grid to represent a nearly homogeneous universe.

### 2. Small Velocity Perturbations
Random Gaussian velocity "kicks" are added to break symmetry and mimic primordial density fluctuations.

> 💡 In more advanced setups, initial displacements would be generated from a cosmological power spectrum using the Zel'dovich approximation or 2LPT.

### 3. Equal Mass Particles
Each particle is assigned the same mass to simplify gravity calculations. In cosmological simulations, this represents dark matter particles.

### Units
Positions are typically in comoving coordinates (e.g., Mpc/h), and velocities are in units consistent with your time integration (e.g., km/s or normalized units).


In [2]:
from scipy.integrate import quad

def integrand(z):
    return (1 + z) / cosmo.efunc(z)**3

def growth_factor(z):
    """Computes linear growth factor D(z), normalized to D(0) = 1."""
    integral, _ = quad(integrand, z, np.inf)
    fz = cosmo.efunc(z) * integral
    f0 = cosmo.efunc(0) * quad(integrand, 0, np.inf)[0]
    return fz / f0

In [13]:
# Simulation parameters
Ngrid = 16             # Number of particles per side (N = Ngrid^3)
box_size = 1.0        # Size of the box in Mpc/h
D = growth_factor(z=100)  # Linear growth factor (normalized)

# 1. Generate displacement field
disp_field = displacement_field(Ngrid, box_size, power_spectrum)

# 2. Generate uniform grid of particle positions
lin = np.linspace(0, box_size, Ngrid, endpoint=False)
X, Y, Z = np.meshgrid(lin, lin, lin, indexing='ij')
grid_pos = np.stack([X, Y, Z], axis=-1)  # Shape: (N, N, N, 3)

# 3. Apply displacements
pos = (grid_pos + D * disp_field).reshape(-1, 3)  # Final positions
vel = (D * disp_field).reshape(-1, 3)             # Initial velocities

# Number of particles
N = Ngrid**3

# 4. Mass assignment
mass = 1000* np.ones((N, 1)) * (cosmo.Om0*cosmo.critical_density0*(box_size*u.Mpc/N)**3).to('Msun').value

## Gravitational Acceleration Calculation

In this section, we compute the gravitational acceleration acting on each particle due to all others using **Newton's Law of Gravitation**.

### Key Concepts:
- Newton's law states that the force between two masses is proportional to the product of their masses and inversely proportional to the square of the distance between them.
- Here, we compute the **acceleration** rather than force, assuming all masses are subject to gravitational influence.
- A **softening length** (ε) is introduced to avoid divergences when particles are very close to each other (i.e., when r → 0).

### Function Overview:
`get_accelleration(pos, mass, G, softening)`

- **Input:**
  - `pos`: an (N×3) array containing the 3D positions of N particles.
  - `mass`: an (N×1) array of masses.
  - `G`: gravitational constant.
  - `softening`: a small number to soften close interactions.

- **Output:**
  - `a`: an (N×3) array of accelerations acting on each particle.

This function builds the foundation of our N-body simulation by determining how each particle responds to the gravitational field generated by others.


In [14]:
def get_accelleration(pos, mass, G, softening):
    """
    Calculate the gravitational acceleration on each particle due to all others
    using Newton's Law of Gravitation.

    Parameters
    ----------
    pos : ndarray of shape (N, 3)
        Particle positions in 3D space.
    mass : ndarray of shape (N, 1)
        Masses of the particles.
    G : float
        Gravitational constant.
    softening : float
        Softening length to avoid numerical instabilities at short distances.

    Returns
    -------
    a : ndarray of shape (N, 3)
        Accelerations acting on each particle in 3D space.
    """
    # Extract x, y, z positions of all particles
    x = pos[:, 0:1]  # shape (N, 1)
    y = pos[:, 1:2]
    z = pos[:, 2:3]

    # Compute pairwise separation vectors: r_j - r_i for each dimension
    dx = x.T - x  # shape (N, N)
    dy = y.T - y
    dz = z.T - z

    # Compute the inverse cube of the distance with softening
    inv_r3 = (dx**2 + dy**2 + dz**2 + softening**2)**(-1.5)

    # Compute the acceleration components using Newton's law
    ax = G * (dx * inv_r3) @ mass  # shape (N, 1)
    ay = G * (dy * inv_r3) @ mass
    az = G * (dz * inv_r3) @ mass

    # Stack the components to get the total acceleration vector
    a = np.hstack((ax, ay, az))  # shape (N, 3)

    return a

# N-body simulation

Simulation parameters and constants

In [15]:
# time start, time-step and time end
t_start, t_end, dt = (0.0*u.Myr).value, (5000*u.Myr).value, (10.0*u.Myr).value

# softening length
softening = (1e-6 *u.Mpc).value

# Newton's Gravitational constant
G_grav = cst.G.to('Mpc^3/(Msun*Myr^2)').value

In [16]:
# Copy to GPU device
mass = cp.ones((N,1))  # in Msun units
vel  = cp.zeros((N,3)) #cp.random.randn(N,3)/50+cp.array([0.0, 100., 0.0])
pos  = cp.array(pos)   # in kpc units
G = cp.array(G_grav)
softening = cp.array(softening)
dt = cp.array(dt)

In [17]:
# calculate initial gravitational accelerations
acc = get_accelleration(pos, mass, G_grav, softening)

# number of timesteps
Nt = int(np.ceil(t_end/dt))

# save particle orbits
pos_save = cp.zeros((N,3,Nt+1))
pos_save[:,:,0] = pos

t_all = cp.arange(Nt+1)*dt
t = t_start

# Simulation Main Loop
for i in tqdm(range(Nt)):
    # (1/2) kick
    vel += acc * dt/2.0

    # drift
    pos += vel * dt

    # update accelerations
    acc = get_accelleration(pos, mass, G_grav, softening)

    # (1/2) kick
    vel += acc * dt/2.0

    # update time
    t += dt

    # save energies, positions for plotting trail
    pos_save[:,:,i+1] = pos

pos_save = pos_save.get()
mass = mass.get()

100%|█████████████████████████████████████████| 500/500 [01:00<00:00,  8.24it/s]


## Plot

In [18]:
path_out = './cosmo_snapshot/'

for i in tqdm(range(Nt)[slice(None,None,10)]):
    #for i in tqdm(range(Nt)):
    fig, ax = plt.subplots(figsize=(15, 15), ncols=1, nrows=1, subplot_kw={'projection': '3d'})
    ax.set_axis_off()
    
    #ax.grid(True, color='grey', alpha=0.2)  # Grid lines with transparency
    ax.set_facecolor('white')
    
    x, y, z = pos_save[...,i].T
    ax.scatter(x, y, z, marker='o', s=10, color='k')
    if(i < 25):
        ax.scatter(pos_save[:,0,:i].T, pos_save[:,1,:i].T, pos_save[:,2,:i].T, marker='.', s=1, color='red', alpha=0.2)
    else:
        ax.scatter(pos_save[:,0,i-25:i].T, pos_save[:,1,i-25:i].T, pos_save[:,2,i-25:i].T, marker='.', s=1, color='red', alpha=0.2)
    
    pos_cm = np.mean(pos_save[...,i]*mass, axis=0)/np.mean(mass)
    #ax.set_xlim(pos_cm[0]-2.0, pos_cm[0]+2.0), ax.set_ylim(pos_cm[1]-2.0, pos_cm[1]+2.0), ax.set_zlim(pos_cm[2]-2.0, pos_cm[2]+2.0)
    
    ax.set_xlabel('x'), ax.set_ylabel('y'), ax.set_zlabel('z')
    ax.set_xlim(-0.1, 1.1), ax.set_ylim(-0.1, 1.1), ax.set_zlim (-0.1, 1.1)
    #ax.view_init(elev=45, azim=45)

    plt.savefig('%ssnapshot_%d.png' %(path_out, i), bbox_inches='tight')
    plt.clf(), plt.close()

100%|███████████████████████████████████████████| 50/50 [03:00<00:00,  3.60s/it]


## Create GIF

In [18]:
def CreateMovie(filename, array, fps=5, scale=1., fmt='avi'):
    ''' Create and save a gif or video from array of images.
        Parameters:
            * filename (string): name of the saved video
            * array (list or string): array of images name already in order, if string it supposed to be the first part of the images name (before iteration integer)
            * fps = 5 (integer): frame per seconds (limit human eye ~ 15)
            * scale = 1. (float): ratio factor to scale image hight and width
            * fmt (string): file extention of the gif/video (e.g: 'gif', 'mp4' or 'avi')
        Return:
            * moviepy clip object
    '''
    if(isinstance(array, str)):
        array = np.array(sorted(glob(array+'*.png'), key=os.path.getmtime))
    else:
        pass
    filename += '.'+fmt
    clip = ImageSequenceClip(list(array), fps=fps).resize(scale)
    if(fmt == 'gif'):
        clip.write_gif(filename, fps=fps)
    elif(fmt == 'mp4'):
        clip.write_videofile(filename, fps=fps, codec='mpeg4')
    elif(fmt == 'avi'):
        clip.write_videofile(filename, fps=fps, codec='png')
    else:
        print('Error! Wrong File extension.')
        sys.exit()

    # print the size of the movie
    command = os.popen('du -sh %s' % filename)
    print(command.read())
    return clip


In [19]:
img_arr = ['%ssnapshot_%d.png' %(path_out, i) for i in range(len(glob(path_out+'*png')))]

#CreateMovie(filename='earth_sun_system', array=img_arr, fps=15, fmt='avi')
CreateMovie(filename='solar_system', array=img_arr, fps=15, fmt='avi')

Moviepy - Building video solar_system.avi.
Moviepy - Writing video solar_system.avi



Moviepy - Done !
Moviepy - video ready solar_system.avi
270M	solar_system.avi



In [21]:
pos_save[3,:,-1]

array([-20.4855863 ,   4.08417744,   0.        ])